In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs


# ============================================================
# 文件路径
# ============================================================

FULL_CSV = r"../outputs/recommendations_after_post_screening_model_based.csv"
RANDOM_CSV = r"../outputs/recommendations_after_post_screening_random.csv"
MORGAN_CSV = r"../outputs/recommendations_after_post_screening_morgan_ema_True_decor_True_topk_True.csv"

WO_EMA_CSV = r"../outputs/recommendations_after_post_screening_full_model_ema_False_decor_True_topk_True.csv"
WO_DECOR_CSV = r"../outputs/recommendations_after_post_screening_full_model_ema_True_decor_False_topk_True.csv"
WO_TOPK_CSV = r"../outputs/recommendations_after_post_screening_full_model_ema_True_decor_True_topk_False.csv"


FULL_EMB = r"../outputs/embeddings_after_post_screening_model_based.npy"
RANDOM_EMB = r"../outputs/embeddings_after_post_screening_random.npy"
MORGAN_EMB = r"../outputs/embeddings_after_post_screening_morgan_ema_True_decor_True_topk_True.npy"

WO_EMA_EMB = r"../outputs/embeddings_after_post_screening_full_model_ema_False_decor_True_topk_True.npy"
WO_DECOR_EMB = r"../outputs/embeddings_after_post_screening_full_model_ema_True_decor_False_topk_True.npy"
WO_TOPK_EMB = r"../outputs/embeddings_after_post_screening_full_model_ema_True_decor_True_topk_False.npy"


PROTO_EMB = r"../checkpoints_origin_backup/proto_centroids_GINE_epoch_300.pth"

OUT_DIR = r"./"
SMILES_COL = "smiles"

FULL_BEFORE_N = 30800
RANDOM_BEFORE_N = 30800
MORGAN_BEFORE_N = 30800
WO_EMA_BEFORE_N = 30800
WO_DECOR_BEFORE_N = 30800
WO_TOPK_BEFORE_N = 30800


# ============================================================
# 辅助函数
# ============================================================

def ensure_out_dir(out_dir):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)


def check_file_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到文件：{path}")


def load_data(csv_path, emb_path):
    check_file_exists(csv_path)
    check_file_exists(emb_path)

    df = pd.read_csv(csv_path)
    emb = np.load(emb_path)

    if len(df) != emb.shape[0]:
        raise ValueError(
            f"CSV 行数和 embedding 数量不一致：\n"
            f"{csv_path}: {len(df)} rows\n"
            f"{emb_path}: {emb.shape[0]} embeddings"
        )

    return df, emb


def load_prototype_centroids(pth_path):
    check_file_exists(pth_path)

    obj = torch.load(pth_path, map_location="cpu")

    if isinstance(obj, torch.Tensor):
        centroids = obj

    elif isinstance(obj, dict):
        possible_keys = [
            "prototype_centroids",
            "prototypes",
            "centroids",
            "proto_centroids",
            "prototype_embeddings",
            "cluster_centers",
            "proto_centroids_GINE"
        ]

        found_key = None
        for key in possible_keys:
            if key in obj:
                found_key = key
                break

        if found_key is None:
            raise KeyError(
                f"在 {pth_path} 中没有找到 prototype centroids。\n"
                f"当前 pth 文件包含的 keys 是：{list(obj.keys())}\n"
                f"请把真实 key 加入 possible_keys。"
            )

        centroids = obj[found_key]

    else:
        raise TypeError(f"不支持的 pth 文件格式：{type(obj)}")

    if isinstance(centroids, torch.Tensor):
        centroids = centroids.detach().cpu().numpy()
    elif isinstance(centroids, np.ndarray):
        pass
    else:
        centroids = np.array(centroids)

    if centroids.ndim != 2:
        raise ValueError(
            f"prototype_centroids 应该是二维数组 [num_prototypes, emb_dim]，"
            f"但现在 shape 是 {centroids.shape}"
        )

    return centroids


# ============================================================
# 指标计算
# ============================================================

def compute_prototype_assignment(candidate_embeddings, prototype_centroids):
    Z = normalize(candidate_embeddings, axis=1)
    C = normalize(prototype_centroids, axis=1)

    sim_matrix = Z @ C.T

    assigned_proto = np.argmax(sim_matrix, axis=1)
    max_sim = np.max(sim_matrix, axis=1)

    sorted_sim = np.sort(sim_matrix, axis=1)

    if sim_matrix.shape[1] >= 2:
        second_sim = sorted_sim[:, -2]
    else:
        second_sim = np.full_like(max_sim, np.nan)

    similarity_gap = max_sim - second_sim

    return sim_matrix, max_sim, assigned_proto, second_sim, similarity_gap


def compute_prototype_coverage(assigned_proto, num_prototypes):
    covered = np.unique(assigned_proto)
    coverage_ratio = len(covered) / num_prototypes
    coverage_text = f"{len(covered)}/{num_prototypes}"

    return coverage_ratio, coverage_text, covered


def compute_assignment_entropy(assigned_proto, num_prototypes):
    counts = np.bincount(assigned_proto, minlength=num_prototypes)

    if counts.sum() == 0:
        probs = np.zeros_like(counts, dtype=float)
        return np.nan, np.nan, counts, probs

    probs = counts / counts.sum()
    probs_nonzero = probs[probs > 0]

    entropy = -np.sum(probs_nonzero * np.log(probs_nonzero))
    norm_entropy = entropy / np.log(num_prototypes) if num_prototypes > 1 else np.nan

    return entropy, norm_entropy, counts, probs


def compute_silhouette(candidate_embeddings, assigned_proto):
    unique_labels = np.unique(assigned_proto)

    if len(unique_labels) < 2 or len(unique_labels) >= len(assigned_proto):
        return np.nan

    Z = normalize(candidate_embeddings, axis=1)

    return silhouette_score(Z, assigned_proto, metric="cosine")


def compute_morgan_fps(smiles_list, radius=2, fp_size=1024):
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=fp_size
    )

    fps = []
    valid_smiles = []
    invalid_smiles = []

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))

        if mol is None:
            fps.append(None)
            invalid_smiles.append(smi)
        else:
            fp = generator.GetFingerprint(mol)
            fps.append(fp)
            valid_smiles.append(smi)

    return fps, valid_smiles, invalid_smiles


def compute_candidate_diversity(smiles_list):
    fps, valid_smiles, invalid_smiles = compute_morgan_fps(smiles_list)

    fps = [fp for fp in fps if fp is not None]

    if len(fps) < 2:
        return np.nan, np.nan, len(valid_smiles), len(invalid_smiles)

    sims = []

    for i in range(len(fps)):
        for j in range(i + 1, len(fps)):
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            sims.append(sim)

    mean_tanimoto_similarity = float(np.mean(sims))
    diversity = 1.0 - mean_tanimoto_similarity

    return diversity, mean_tanimoto_similarity, len(valid_smiles), len(invalid_smiles)


def evaluate_method(
    method_name,
    df,
    embeddings,
    prototype_centroids,
    smiles_col,
    before_screening_n=None
):
    num_prototypes = prototype_centroids.shape[0]

    sim_matrix, max_sim, assigned_proto, second_sim, similarity_gap = compute_prototype_assignment(
        embeddings,
        prototype_centroids
    )

    coverage_ratio, coverage_text, covered = compute_prototype_coverage(
        assigned_proto,
        num_prototypes
    )

    entropy, norm_entropy, proto_counts, proto_probs = compute_assignment_entropy(
        assigned_proto,
        num_prototypes
    )

    silhouette = compute_silhouette(
        embeddings,
        assigned_proto
    )

    if smiles_col not in df.columns:
        raise ValueError(
            f"CSV 中找不到 SMILES 列：{smiles_col}\n"
            f"当前 CSV 列名为：{list(df.columns)}"
        )

    smiles_list = df[smiles_col].astype(str).tolist()

    diversity, mean_tanimoto, valid_smiles_num, invalid_smiles_num = compute_candidate_diversity(
        smiles_list
    )

    survival_rate = len(df) / before_screening_n if before_screening_n else np.nan

    summary = {
        "Method": method_name,
        "Before screening number": before_screening_n if before_screening_n is not None else np.nan,
        "After screening number": len(df),
        "Post-screening survival rate": survival_rate,

        "Prototype similarity mean": float(np.mean(max_sim)),
        "Prototype similarity std": float(np.std(max_sim)),
        "Prototype similarity median": float(np.median(max_sim)),
        "Prototype similarity min": float(np.min(max_sim)),
        "Prototype similarity max": float(np.max(max_sim)),

        "Similarity gap mean": float(np.mean(similarity_gap)),
        "Similarity gap std": float(np.std(similarity_gap)),

        "Prototype coverage": coverage_text,
        "Prototype coverage ratio": float(coverage_ratio),

        "Assignment entropy": float(entropy),
        "Normalized assignment entropy": float(norm_entropy),

        "Candidate diversity": float(diversity),
        "Mean pairwise Tanimoto similarity": float(mean_tanimoto),

        "Silhouette score": float(silhouette),

        "Valid SMILES number": valid_smiles_num,
        "Invalid SMILES number": invalid_smiles_num,
    }

    for i in range(num_prototypes):
        summary[f"Prototype {i + 1} count"] = int(proto_counts[i])
        summary[f"Prototype {i + 1} ratio"] = float(proto_probs[i])

    detail_df = df.copy()
    detail_df["method"] = method_name
    detail_df["assigned_prototype"] = assigned_proto + 1
    detail_df["max_prototype_similarity"] = max_sim
    detail_df["second_prototype_similarity"] = second_sim
    detail_df["similarity_gap"] = similarity_gap

    for i in range(num_prototypes):
        detail_df[f"prototype_{i + 1}_similarity"] = sim_matrix[:, i]

    return summary, detail_df


def make_paper_table(summary_df):
    paper_df = summary_df.copy()

    paper_df["Prototype similarity ↑"] = paper_df.apply(
        lambda row: f"{row['Prototype similarity mean']:.4f} ± {row['Prototype similarity std']:.4f}",
        axis=1
    )

    paper_df["Prototype coverage ↑"] = paper_df["Prototype coverage"]

    paper_df["Assignment entropy ↑"] = paper_df["Normalized assignment entropy"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["Candidate diversity ↑"] = paper_df["Candidate diversity"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["Post-screening survival ↑"] = paper_df["Post-screening survival rate"].apply(
        lambda x: f"{x * 100:.4f}%" if pd.notna(x) else "nan"
    )

    paper_df["Silhouette ↑"] = paper_df["Silhouette score"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["After screening number"] = paper_df["After screening number"].astype(int)

    final_table = paper_df[
        [
            "Method",
            "After screening number",
            "Prototype similarity ↑",
            "Prototype coverage ↑",
            "Assignment entropy ↑",
            "Candidate diversity ↑",
            "Post-screening survival ↑",
            "Silhouette ↑"
        ]
    ]

    return final_table


def save_detail_csv(detail_df, method_name):
    safe_name = (
        method_name
        .lower()
        .replace(" ", "_")
        .replace("/", "without_")
        .replace("-", "_")
    )

    out_path = os.path.join(
        OUT_DIR,
        f"{safe_name}_postscreen_with_metrics.csv"
    )

    detail_df.to_csv(out_path, index=False)


# ============================================================
# 主程序
# ============================================================

def main():
    ensure_out_dir(OUT_DIR)

    print("Loading prototype centroids...")
    prototype_centroids = load_prototype_centroids(PROTO_EMB)
    print("Prototype centroids shape:", prototype_centroids.shape)

    # 所有方法统一放在这里，后面想继续加 ablation 也很方便
    methods = [
        {
            "name": "Full model",
            "csv": FULL_CSV,
            "emb": FULL_EMB,
            "before_n": FULL_BEFORE_N,
        },
        {
            "name": "Random selection",
            "csv": RANDOM_CSV,
            "emb": RANDOM_EMB,
            "before_n": RANDOM_BEFORE_N,
        },
        {
            "name": "Morgan fingerprints",
            "csv": MORGAN_CSV,
            "emb": MORGAN_EMB,
            "before_n": MORGAN_BEFORE_N,
        },
        {
            "name": "w/o EMA",
            "csv": WO_EMA_CSV,
            "emb": WO_EMA_EMB,
            "before_n": WO_EMA_BEFORE_N,
        },
        {
            "name": "w/o decorrelation",
            "csv": WO_DECOR_CSV,
            "emb": WO_DECOR_EMB,
            "before_n": WO_DECOR_BEFORE_N,
        },
        {
            "name": "w/o top-K",
            "csv": WO_TOPK_CSV,
            "emb": WO_TOPK_EMB,
            "before_n": WO_TOPK_BEFORE_N,
        },
    ]

    all_summaries = []
    all_details = []

    print("\nEvaluating methods...")

    for item in methods:
        method_name = item["name"]
        csv_path = item["csv"]
        emb_path = item["emb"]
        before_n = item["before_n"]

        print(f"\nLoading {method_name}...")
        df, embeddings = load_data(csv_path, emb_path)

        # print(f"{method_name} CSV shape:", df.shape)
        # print(f"{method_name} embedding shape:", embeddings.shape)

        if embeddings.shape[1] != prototype_centroids.shape[1]:
            raise ValueError(
                f"{method_name} embedding dim 和 prototype dim 不一致："
                f"{embeddings.shape[1]} vs {prototype_centroids.shape[1]}"
            )

        summary, detail_df = evaluate_method(
            method_name=method_name,
            df=df,
            embeddings=embeddings,
            prototype_centroids=prototype_centroids,
            smiles_col=SMILES_COL,
            before_screening_n=before_n
        )

        all_summaries.append(summary)
        all_details.append(detail_df)

        save_detail_csv(detail_df, method_name)

    summary_df = pd.DataFrame(all_summaries)
    paper_table = make_paper_table(summary_df)

    # 保存 summary
    summary_df.to_csv(
        os.path.join(OUT_DIR, "postscreen_ablation_summary.csv"),
        index=False
    )

    summary_df.to_excel(
        os.path.join(OUT_DIR, "postscreen_ablation_summary.xlsx"),
        index=False
    )

    # 保存主文表格
    paper_table.to_csv(
        os.path.join(OUT_DIR, "main_text_ablation_table.csv"),
        index=False
    )

    paper_table.to_excel(
        os.path.join(OUT_DIR, "main_text_ablation_table.xlsx"),
        index=False
    )

    # 保存所有 detail 合并文件
    all_detail_df = pd.concat(all_details, axis=0, ignore_index=True)

    all_detail_df.to_csv(
        os.path.join(OUT_DIR, "all_methods_postscreen_with_metrics.csv"),
        index=False
    )

    print("\nDone! Main-text ablation table:")
    print(paper_table.to_string(index=False))

    # print("\nSaved files:")
    # print(os.path.join(OUT_DIR, "postscreen_ablation_summary.csv"))
    # print(os.path.join(OUT_DIR, "postscreen_ablation_summary.xlsx"))
    # print(os.path.join(OUT_DIR, "main_text_ablation_table.csv"))
    # print(os.path.join(OUT_DIR, "main_text_ablation_table.xlsx"))
    # print(os.path.join(OUT_DIR, "all_methods_postscreen_with_metrics.csv"))


if __name__ == "__main__":
    main()

Loading prototype centroids...
Prototype centroids shape: (7, 128)

Evaluating methods...

Loading Full model...

Loading Random selection...

Loading Morgan fingerprints...

Loading w/o EMA...

Loading w/o decorrelation...

Loading w/o top-K...

Done! Main-text ablation table:
             Method  After screening number Prototype similarity ↑ Prototype coverage ↑ Assignment entropy ↑ Candidate diversity ↑ Post-screening survival ↑ Silhouette ↑
         Full model                      70        0.7907 ± 0.0300                  5/7               0.7229                0.8904                   0.2273%       0.9762
   Random selection                     647        0.5217 ± 0.1478                  7/7               0.5297                0.7775                   2.1006%       0.5965
Morgan fingerprints                    1598        0.4971 ± 0.1772                  7/7               0.8369                0.7417                   5.1883%       0.4084
            w/o EMA                    17